# MotionGPT3: テキストから 3D モーションを生成して再生する（Google Colab 版）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yamak493/MotionGPT3/blob/main/notebooks/MotionGPT3_text2motion_colab.ipynb)

英語の文章（例: *a person walks forward and waves*）から人の 3D モーションを生成し、ノートブック上でアニメーションとして再生します。

- **学習はしません。** 学習済みの重みをダウンロードして推論だけ行います。
- 重みは、公式 MotionGPT3 の最終チェックポイントを [Motius](https://github.com/ZeyuLing/Motius)（旧 hftrainer）が推論に必要な一式（チェックポイント・設定ファイル・GPT-2・正規化統計）としてまとめ直した [`ZeyuLing/Motius-MotionGPT3-HumanML3D`](https://huggingface.co/ZeyuLing/Motius-MotionGPT3-HumanML3D)を使います（hftrainer 時代の名前と思われる `ZeyuLing/hftrainer-motiongpt3-humanml3d` も、取得できない場合の予備として試します）。
- 推論コードは Motius パッケージ（MotionGPT3 公式コードを同梱）を使うので、このリポジトリのセットアップ（SMPL・データセット・評価器のダウンロードなど）は不要です。
- 出力は HumanML3D 形式（263 次元特徴量・20fps・22 関節）です。関節座標に変換して骨格アニメーションで表示します。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選びます（無料枠の T4 で動きます。**GPU は必須** で、CPU では動きません）。
2. 上のセルから順に実行します（**ランタイム → すべてのセルを実行** でも OK）。
3. 「6. テキストからモーションを生成」のフォームでプロンプトを変えて、そのセルだけ再実行すれば何度でも生成できます。

| 項目 | 目安 |
| --- | --- |
| GPU | T4 (16GB) で十分。モデルは約 238M パラメータ |
| 初回準備 | セットアップ + 重みのダウンロードで数分 |
| 生成 | 1 本あたり数秒〜十数秒（拡散ヘッド 100 ステップ） |
| 長さ | 2.0〜9.8 秒（40〜196 フレーム、4 フレーム単位） |

## 1. GPU の確認

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU が見つかりません。メニューの「ランタイム → ランタイムのタイプを変更」で T4 GPU を選び、"
        "もう一度上から実行してください（MotionGPT3 の推論コードは GPU が前提です）。"
    )
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. セットアップ（Motius のインストール）

Motius（旧 hftrainer）を GitHub から取得してインストールします。このノートブック作成時点のコミットに固定しています（`MOTIUS_REF` を `"main"` にすると最新版を使えます）。

Motius には 3D アセットなどの大きな Git LFS ファイルも含まれますが、MotionGPT3 の推論には不要なのでダウンロードを省略しています（`GIT_LFS_SKIP_SMUDGE=1`）。

In [ ]:
import os
import sys

MOTIUS_DIR = "/content/Motius"
MOTIUS_REF = "8ace07573153067079358e6da34e4377bc951001"  # ノートブック作成時点のコミット（"main" で最新版）

os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"  # LFS の大きなファイルは取得しない
if not os.path.isdir(os.path.join(MOTIUS_DIR, "motius")):
    !rm -rf {MOTIUS_DIR}
    !git init -q {MOTIUS_DIR}
    !git -C {MOTIUS_DIR} remote add origin https://github.com/ZeyuLing/Motius.git
    !git -C {MOTIUS_DIR} fetch -q --depth 1 origin {MOTIUS_REF}
    !git -C {MOTIUS_DIR} -c filter.lfs.smudge= -c filter.lfs.process= -c filter.lfs.required=false checkout -q FETCH_HEAD
!git -C {MOTIUS_DIR} log -1 --format="Motius commit: %h (%cd)"

# MotionGPT3 用の依存パッケージ込みでインストール（Colab に入っているものはそのまま使われます）
!pip install -q -e "{MOTIUS_DIR}[motiongpt3]"

# editable インストールはランタイム再起動まで import できないことがあるので、パスを直接通す
if MOTIUS_DIR not in sys.path:
    sys.path.insert(0, MOTIUS_DIR)
print("✅ セットアップ完了")

## 3. 学習済み重みのダウンロード

Hugging Face Hub から重み一式をダウンロードします（数分かかります）。公開リポジトリなので **トークンは不要** です（`HF_TOKEN` が無いという警告が出ても無視して大丈夫です）。

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

# Motius の現行名 → hftrainer 時代の名前（予備）の順に試す
REPO_CANDIDATES = [
    "ZeyuLing/Motius-MotionGPT3-HumanML3D",
    "ZeyuLing/hftrainer-motiongpt3-humanml3d",
]
WEIGHTS_DIR = Path("/content/motiongpt3_weights")

for repo_id in REPO_CANDIDATES:
    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=str(WEIGHTS_DIR),
            ignore_patterns=["*.mp4", "*.gif", "*.png", "*.jpg", "*.webm"],  # 推論に不要なメディア
        )
        print("✅ ダウンロード元:", repo_id)
        break
    except Exception as e:
        print(f"⚠️ {repo_id} から取得できませんでした: {type(e).__name__}: {e}")
else:
    raise RuntimeError("重みをダウンロードできませんでした。ネットワークやリポジトリ名を確認してください。")

required = ["motiongpt3.ckpt", "configs/test.yaml", "deps/mot-gpt2/config.json", "assets/meta/mean.npy", "assets/meta/std.npy"]
missing = [p for p in required if not (WEIGHTS_DIR / p).exists()]
if missing:
    raise FileNotFoundError(f"重みフォルダに必要なファイルがありません: {missing}")
total_gb = sum(f.stat().st_size for f in WEIGHTS_DIR.rglob("*") if f.is_file()) / 1024**3
print(f"保存先: {WEIGHTS_DIR}（合計 {total_gb:.2f} GB）")

## 4. モデルの読み込み

少し時間がかかります。途中で `guidance_scale 3.0` などのログが表示されますが正常です。

In [ ]:
import importlib
import os
import pickle
import torch
from motius.models.motiongpt3 import bundle as mgpt3_bundle
from motius.pipelines.motiongpt3 import MotionGPT3Pipeline

DEVICE = "cuda"  # 推論コード内部で .cuda() を直接呼んでいるため GPU 必須

# 学習時の評価指標モジュールは推論に不要で、別途の評価器ファイルを要求するため無効化する。
# Motius も同じ処理をしているが、設定ファイルが旧来の `motGPT.` パスを指す場合にも効くよう両方に適用する。
mgpt3_bundle._install_legacy_aliases()
for _name in ("motius.models.motiongpt3.network.motGPT.models.base", "motGPT.models.base"):
    importlib.import_module(_name).BaseModel.configure_metrics = (
        lambda self: setattr(self, "metrics", torch.nn.Module()))


def load_pipeline():
    return MotionGPT3Pipeline.from_pretrained(
        str(WEIGHTS_DIR),
        bundle_kwargs={
            "device": DEVICE,
            "guidance_scale": 7.5,  # 論文・公式設定のデフォルト
            "runtime_dir": "/content/motiongpt3_runtime",
        },
    )


try:
    pipe = load_pipeline()
except pickle.UnpicklingError:
    # PyTorch 2.6 以降は torch.load が既定で weights_only=True になり、
    # PyTorch Lightning 形式のチェックポイントが読めないことがある。
    # 配布元を信頼できる場合に限り、従来どおりの読み込みに切り替える。
    print("weights_only での読み込みに失敗したため、従来方式で読み込み直します")
    os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
    pipe = load_pipeline()

print("✅ 読み込み完了 | device:", pipe.device, "| load report:", pipe.bundle.load_report)
if torch.cuda.is_available():
    print(f"VRAM 使用量: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 5. プレビュー用の関数

生成された HumanML3D-263 特徴量を 22 関節の 3D 座標に変換し、骨格アニメーション（MP4）を作る関数です。実行するだけで OK です。

In [ ]:
import base64
import os
import shutil

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# HumanML3D の 22 関節の骨格（SMPL の関節番号）
KINEMATIC_CHAINS = [
    [0, 2, 5, 8, 11],      # 右脚
    [0, 1, 4, 7, 10],      # 左脚
    [0, 3, 6, 9, 12, 15],  # 体幹〜頭
    [9, 14, 17, 19, 21],   # 右腕
    [9, 13, 16, 18, 20],   # 左腕
]
CHAIN_COLORS = ["#d62728", "#1f77b4", "#444444", "#d62728", "#1f77b4"]  # 右=赤, 左=青
FPS = 20  # HumanML3D は 20fps


def _qinv(q):
    q = q.copy()
    q[..., 1:] *= -1
    return q


def _qrot(q, v):
    qvec = q[..., 1:]
    uv = np.cross(qvec, v)
    uuv = np.cross(qvec, uv)
    return v + 2 * (q[..., :1] * uv + uuv)


def hml263_to_joints(feats, joints_num=22):
    """HumanML3D-263 特徴量 (T, 263) -> 関節座標 (T, 22, 3)。HumanML3D の recover_from_ric と同じ処理。"""
    feats = np.asarray(feats, dtype=np.float32)
    T = feats.shape[0]
    # ルートの向き（Y 軸回り）を角速度から積分
    r_rot_ang = np.zeros(T, dtype=np.float32)
    r_rot_ang[1:] = feats[:-1, 0]
    r_rot_ang = np.cumsum(r_rot_ang)
    r_rot_quat = np.zeros((T, 4), dtype=np.float32)
    r_rot_quat[:, 0] = np.cos(r_rot_ang)
    r_rot_quat[:, 2] = np.sin(r_rot_ang)
    # ルート位置を速度から積分
    r_pos = np.zeros((T, 3), dtype=np.float32)
    r_pos[1:, [0, 2]] = feats[:-1, 1:3]
    r_pos = _qrot(_qinv(r_rot_quat), r_pos)
    r_pos = np.cumsum(r_pos, axis=0)
    r_pos[:, 1] = feats[:, 3]
    # ルート相対の関節位置をワールド座標へ
    pos = feats[:, 4:(joints_num - 1) * 3 + 4].reshape(T, joints_num - 1, 3)
    q = np.broadcast_to(_qinv(r_rot_quat)[:, None, :], pos.shape[:-1] + (4,))
    pos = _qrot(q, pos)
    pos[..., 0] += r_pos[:, None, 0]
    pos[..., 2] += r_pos[:, None, 2]
    return np.concatenate([r_pos[:, None, :], pos], axis=1)


def to_plot_coords(joints):
    """HumanML3D (Y-up) -> 描画用 (Z-up)。鏡像にならないよう (x, -z, y) に回転し、床を z=0 に合わせる。"""
    P = np.stack([joints[..., 0], -joints[..., 2], joints[..., 1]], axis=-1)
    P[..., 2] -= P[..., 2].min()
    return P


def _ffmpeg_path():
    path = shutil.which("ffmpeg")
    if path is None:
        import imageio_ffmpeg
        path = imageio_ffmpeg.get_ffmpeg_exe()
    return path


def render_skeleton_mp4(joints, out_path, title="", fps=FPS, radius=1.1, size_px=512):
    """関節座標 (T, 22, 3) から、人物を追従するカメラの骨格アニメーション MP4 を作る。"""
    plt.rcParams["animation.ffmpeg_path"] = _ffmpeg_path()
    P = to_plot_coords(joints)
    T = P.shape[0]
    root = P[:, 0, :2]
    grid = np.arange(np.floor(root.min() - radius * 2), np.ceil(root.max() + radius * 2) + 1e-6, 0.5)

    dpi = 100
    fig = plt.figure(figsize=(size_px / dpi, size_px / dpi), dpi=dpi)
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    ax = fig.add_subplot(111, projection="3d")
    wrapped = "\n".join(title[i:i + 60] for i in range(0, len(title), 60))

    def update(t):
        ax.cla()
        cx, cy = root[t]
        x0, x1, y0, y1 = cx - radius, cx + radius, cy - radius, cy + radius
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        ax.set_zlim(0, 2 * radius)
        ax.set_box_aspect((1, 1, 1), zoom=1.35)
        ax.view_init(elev=15, azim=-70)
        ax.set_axis_off()
        # 床のグリッド（表示範囲内だけ描く）
        for g in grid:
            if x0 <= g <= x1:
                ax.plot([g, g], [y0, y1], [0, 0], color="#cccccc", lw=0.8)
            if y0 <= g <= y1:
                ax.plot([x0, x1], [g, g], [0, 0], color="#cccccc", lw=0.8)
        # ルートの軌跡（表示範囲内だけ）
        trail = root[: t + 1]
        inside = (trail[:, 0] >= x0) & (trail[:, 0] <= x1) & (trail[:, 1] >= y0) & (trail[:, 1] <= y1)
        ax.plot(trail[inside, 0], trail[inside, 1], np.zeros(inside.sum()), color="#2ca02c", lw=1.5)
        # 骨格
        for chain, color in zip(KINEMATIC_CHAINS, CHAIN_COLORS):
            ax.plot(P[t, chain, 0], P[t, chain, 1], P[t, chain, 2], color=color, lw=3.5,
                    marker="o", markersize=2.5)
        ax.set_title(f"{wrapped}\nframe {t + 1}/{T}", fontsize=9, y=0.95)
        return []

    anim = animation.FuncAnimation(fig, update, frames=T, interval=1000 / fps)
    writer = animation.FFMpegWriter(fps=fps, codec="libx264", extra_args=["-pix_fmt", "yuv420p"])
    anim.save(out_path, writer=writer, dpi=dpi)
    plt.close(fig)
    return out_path


def show_video(path, width=512):
    """MP4 をノートブック上で再生（ループ再生）。"""
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop muted playsinline>'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
    ))


def show_interactive(joints, title="", fps=FPS):
    """Plotly でマウス操作できる 3D ビューア（再生ボタン・スライダー付き）。"""
    import plotly.graph_objects as go

    P = to_plot_coords(joints)
    T = P.shape[0]

    def bones(t):
        return [
            go.Scatter3d(x=P[t, c, 0], y=P[t, c, 1], z=P[t, c, 2], mode="lines+markers",
                         line=dict(color=col, width=8), marker=dict(size=3, color=col),
                         hoverinfo="skip", showlegend=False)
            for c, col in zip(KINEMATIC_CHAINS, CHAIN_COLORS)
        ]

    n_bones = len(KINEMATIC_CHAINS)
    lo = P.reshape(-1, 3).min(0)
    hi = P.reshape(-1, 3).max(0)
    center = (lo + hi) / 2
    half = max((hi - lo).max() / 2, 1.0) + 0.2
    rng = [(center[i] - half, center[i] + half) for i in range(3)]
    rng[2] = (0, 2 * half)
    root = P[:, 0]
    traj = go.Scatter3d(x=root[:, 0], y=root[:, 1], z=np.zeros(T), mode="lines",
                        line=dict(color="#2ca02c", width=3), hoverinfo="skip", showlegend=False)

    frames = [go.Frame(data=bones(t), traces=list(range(n_bones)), name=str(t)) for t in range(T)]
    fig = go.Figure(data=bones(0) + [traj], frames=frames)
    frame_ms = 1000 / fps
    fig.update_layout(
        title=title,
        height=600,
        margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(
            xaxis=dict(range=rng[0], title="x"),
            yaxis=dict(range=rng[1], title="y"),
            zaxis=dict(range=rng[2], title="up"),
            aspectmode="cube",
            camera=dict(eye=dict(x=0.6, y=-1.6, z=0.5)),
        ),
        updatemenus=[dict(
            type="buttons", showactive=False, x=0.05, y=0.05, xanchor="left", yanchor="bottom",
            buttons=[
                dict(label="▶ 再生", method="animate",
                     args=[None, dict(frame=dict(duration=frame_ms, redraw=True),
                                      transition=dict(duration=0), fromcurrent=True, mode="immediate")]),
                dict(label="⏸ 停止", method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
            ],
        )],
        sliders=[dict(
            x=0.2, y=0.02, len=0.75, currentvalue=dict(prefix="frame "),
            steps=[dict(method="animate", label=str(t),
                        args=[[str(t)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
                   for t in range(T)],
        )],
    )
    fig.show()

## 6. テキストからモーションを生成して再生

- **prompt** は英語で書いてください（HumanML3D の英語キャプションで学習されています）。1〜2 文の短めの説明が向いています。
- **seconds** は 2.0〜9.8 秒です（20fps、4 フレーム単位に丸めます）。
- **seed** を変えると別のバリエーションが出ます。同じ seed なら基本的に同じ結果になります。
- **guidance_scale** を上げるとテキストへの忠実度が上がり、下げると多様性が増えます（既定 7.5）。

結果は `/content/outputs/` に保存されます（特徴量 `motion_263.npy`・関節座標 `joints.npy`・プレビュー `preview.mp4`）。

In [ ]:
#@title テキスト → モーション
import json
import random
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import torch

prompt = "a person walks forward and then waves with the right hand."  #@param {type:"string"}
seconds = 5.0  #@param {type:"slider", min:2.0, max:9.8, step:0.2}
seed = 42  #@param {type:"integer"}
guidance_scale = 7.5  #@param {type:"slider", min:1.0, max:12.0, step:0.5}

OUT_ROOT = Path("/content/outputs")


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def generate(prompts, seconds_list, seed=42, guidance_scale=7.5):
    """複数のプロンプトをまとめて生成し、(特徴量, 関節座標) のリストを返す。"""
    lengths = [pipe.clamp_length(round(s * FPS)) for s in seconds_list]
    pipe.bundle.model.guidance_scale = float(guidance_scale)
    set_seed(seed)
    feats = pipe.infer_text_to_motion(list(prompts), lengths)
    return [(f, hml263_to_joints(f)) for f in feats]


def save_result(prompt, feats, joints, seed, guidance_scale):
    slug = re.sub(r"[^a-z0-9]+", "_", prompt.lower()).strip("_")[:40] or "motion"
    out_dir = OUT_ROOT / f"{datetime.now():%Y%m%d_%H%M%S}_{slug}_seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "motion_263.npy", feats)   # HumanML3D-263 特徴量 (T, 263)
    np.save(out_dir / "joints.npy", joints)      # 関節座標 (T, 22, 3), Y-up, メートル
    meta = dict(prompt=prompt, seed=seed, guidance_scale=guidance_scale,
                frames=int(len(feats)), fps=FPS, seconds=len(feats) / FPS)
    (out_dir / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    render_skeleton_mp4(joints, str(out_dir / "preview.mp4"), title=prompt)
    return out_dir


t0 = time.time()
feats, joints = generate([prompt], [seconds], seed=seed, guidance_scale=guidance_scale)[0]
t_gen = time.time() - t0
out_dir = save_result(prompt, feats, joints, seed, guidance_scale)
print(f"生成: {t_gen:.1f} 秒 | {len(feats)} フレーム ({len(feats) / FPS:.1f} 秒) | 保存先: {out_dir}")
show_video(str(out_dir / "preview.mp4"))
last_joints, last_prompt = joints, prompt

## 7. （任意）マウスで回せる 3D ビューア

直前に生成したモーションを Plotly で表示します。ドラッグで視点を回転、ホイールでズームでき、▶ ボタンで再生します。

In [ ]:
show_interactive(last_joints, title=last_prompt)

## 8. （任意）複数のプロンプトをまとめて生成

リストを書き換えて実行してください。バッチでまとめて推論するので 1 本ずつより効率的です。

In [ ]:
batch_prompts = [
    "a person jumps up and down twice.",
    "a person does a cartwheel.",
    "a person sits down on a chair and then stands up.",
    "a person kicks with the left leg.",
]
batch_seconds = [4.0, 5.0, 7.0, 3.0]
batch_seed = 0

results = generate(batch_prompts, batch_seconds, seed=batch_seed, guidance_scale=7.5)
for p, (f, j) in zip(batch_prompts, results):
    d = save_result(p, f, j, batch_seed, 7.5)
    print(f"▶ {p}  ({len(f) / FPS:.1f} 秒)")
    show_video(str(d / "preview.mp4"), width=384)

## 9. 結果のダウンロード

`/content/outputs` を zip にまとめてダウンロードします。`joints.npy` は `(フレーム数, 22, 3)` の関節座標（Y 軸が上、単位はメートル、20fps）です。

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/motiongpt3_outputs", "zip", "/content/outputs")
print("作成:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Colab 以外の環境では上記パスから取得してください。")

## メモ・トラブルシューティング

- **GPU メモリ**: パラメータは約 238M（fp32 で約 1GB）なので、T4（16GB）で十分です。実際の使用量はセル 4 の最後に表示されます。
- **`CUDA out of memory`**: 「8. まとめて生成」の本数を減らすか、ランタイムを再起動してください。
- **重みのダウンロードが失敗する**: Hugging Face の一時的な不調のことがあります。少し待ってセル 3 を再実行してください。
- **最新の Motius を使いたい**: セル 2 の `MOTIUS_REF = "main"` にして、`/content/Motius` を削除してから再実行してください（固定コミットから変わるため、動かなくなる可能性があります）。
- **日本語のプロンプト**: 学習データが英語なので、英語に翻訳してから入力してください。
- **SMPL メッシュでの表示**: このノートブックは骨格（関節）表示のみです。SMPL メッシュ化はこのリポジトリの `fit.py`（SMPL モデルのダウンロードが必要）を参照してください。

## クレジット・ライセンス

- MotionGPT3: Zhu et al., *MotionGPT3: Human Motion as a Second Modality*, 2025（[論文](https://arxiv.org/abs/2506.24086) / [公式コード](https://github.com/OpenMotionLab/MotionGPT3)）
- 推論パイプラインと重みのパッケージ: [Motius](https://github.com/ZeyuLing/Motius)（旧 hftrainer）
- 学習データ HumanML3D・GPT-2 など、各コンポーネントのライセンスに従って利用してください。